# LangGraph Workflow Visualization

This notebook demonstrates the Fihris Agent's LangGraph workflow for generating and validating product descriptions.

## Workflow Overview

The agent follows a multi-step process:
1. **Writer Node**: Generates product copy using Gemini
2. **Validator Node**: Validates the generated content against SEO rules
3. **Routing Logic**: Decides whether to publish, retry, or give up
4. **Publisher Node**: Publishes validated content to WooCommerce
5. **Give Up Node**: Handles max retry scenarios


## 1. Install Dependencies

Make sure you have the required packages installed:

In [ ]:
# Install required packages if not already present
import subprocess
import sys

packages = ['langgraph', 'pydantic', 'beautifulsoup4', 'pygraphviz']
for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

## 2. Define the Workflow Components

In [ ]:
import re
from typing import Dict, Any, Optional, List
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

# Define the AgentState
class AgentState(TypedDict):
    """State dictionary passed through LangGraph nodes."""
    input_data: Dict[str, Any]
    generated_description: Any
    validation_passed: bool
    validation_error: str
    wordpress_product_id: Optional[int]
    wordpress_product_url: Optional[str]
    internal_link_url: Optional[str]
    writer_attempts: int

# Configuration constants
MAX_WRITER_ATTEMPTS = 5
MIN_KEYPHRASE_DENSITY = 0.5
MAX_KEYPHRASE_DENSITY = 3.0
MIN_META_DESC_LEN = 120
MAX_META_DESC_LEN = 156

print("✓ State and configuration defined")

## 3. Define Workflow Nodes

In [ ]:
def writer_node(state: AgentState) -> Dict[str, Any]:
    """
    Writer Node: Generates product copy.
    In production, this calls Gemini API.
    """
    data = state["input_data"]
    
    # In this demo, we return a mock response
    # In production, this would call generate_product_copy()
    mock_response = {
        "description": f"<p>This is a high-quality {data.get('product_name', 'product')} designed for optimal performance.</p>",
        "seo_title": f"{data.get('focus_keyphrase', 'Product')} - Premium Quality",
        "meta_description": f"Discover our {data.get('focus_keyphrase', 'product')}. Premium quality, competitive pricing.",
        "slug": data.get('product_name', 'product').lower().replace(' ', '-'),
        "image_alt": f"High-quality {data.get('product_name', 'product')}",
        "keyphrase_synonyms": ["alternative term 1", "alternative term 2"],
        "attributes": []
    }
    
    return {
        "generated_description": mock_response,
        "validation_passed": False,
        "validation_error": "",
        "internal_link_url": state.get("internal_link_url", "https://example.com"),
        "writer_attempts": state.get("writer_attempts", 0) + 1
    }

def validator_node(state: AgentState) -> Dict[str, Any]:
    """
    Validator Node: Validates generated content against SEO rules.
    """
    # Simplified validation for demo
    generated = state["generated_description"]
    
    if not isinstance(generated, dict):
        return {"validation_passed": False, "validation_error": "Invalid output format"}
    
    # In this demo, we'll pass validation
    # In production, there are ~15 validation checks
    return {"validation_passed": True, "validation_error": ""}

def publisher_node(state: AgentState) -> Dict[str, Any]:
    """
    Publisher Node: Publishes validated content to WooCommerce.
    """
    raw_desc = state.get("generated_description", {})
    input_data = state.get("input_data", {})
    
    # In production, this would call publish_to_woocommerce()
    mock_result = {
        "id": 12345,
        "url": f"https://example.com/product/{input_data.get('product_name', 'product').lower().replace(' ', '-')}"
    }
    
    return {
        "wordpress_product_id": mock_result["id"],
        "wordpress_product_url": mock_result["url"],
        "generated_description": raw_desc
    }

def give_up_node(state: AgentState) -> Dict[str, Any]:
    """
    Give Up Node: Handles max retry scenarios.
    """
    return {
        "validation_error": f"Exceeded {MAX_WRITER_ATTEMPTS} generation attempts. Last error: {state.get('validation_error', '')}"
    }

def routing_logic(state: AgentState) -> str:
    """
    Routing Logic: Decides the next node based on validation status.
    """
    if state["validation_passed"]:
        return "publisher"
    if state.get("writer_attempts", 0) >= MAX_WRITER_ATTEMPTS:
        return "give_up"
    return "writer"

print("✓ All nodes defined")

## 4. Build the Workflow Graph

In [ ]:
# Create the workflow graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("writer", writer_node)
workflow.add_node("validator", validator_node)
workflow.add_node("publisher", publisher_node)
workflow.add_node("give_up", give_up_node)

# Set entry point
workflow.set_entry_point("writer")

# Add edges
workflow.add_edge("writer", "validator")

# Add conditional edges from validator
workflow.add_conditional_edges(
    "validator",
    routing_logic,
    {
        "publisher": "publisher",
        "writer": "writer",
        "give_up": "give_up"
    }
)

# End edges
workflow.add_edge("publisher", END)
workflow.add_edge("give_up", END)

# Compile the workflow
fihris_agent = workflow.compile()

print("✓ Workflow compiled successfully")

## 5. Visualize the Workflow

This will display the graph structure with nodes and edges.

In [ ]:
# Display the workflow graph
try:
    # Try to get the graph drawing
    graph_image = fihris_agent.get_graph().draw_mermaid_png()
    
    from IPython.display import Image, display
    display(Image(data=graph_image))
    print("✓ Graph visualization displayed")
except Exception as e:
    print(f"Note: Could not render graph image ({e}). Showing ASCII representation instead:")
    print()
    print(fihris_agent.get_graph().draw_ascii())

## 6. Display Workflow Structure

In [ ]:
# Print workflow structure
print("WORKFLOW STRUCTURE")
print("=" * 60)
print()
print("Entry Point: writer")
print()
print("Node Descriptions:")
print("-" * 60)
print("1. writer:")
print("   - Generates product copy")
print("   - Calls Gemini API with product specs")
print("   - Outputs: description, SEO title, meta description, slug, image alt")
print()
print("2. validator:")
print("   - Validates 15+ SEO rules")
print("   - Checks keyphrase density, distribution, placement")
print("   - Validates internal/external links")
print("   - Routes to publisher (pass) or retry (fail)")
print()
print("3. publisher:")
print("   - Publishes to WooCommerce API")
print("   - Uploads images and assigns categories")
print("   - Records keyphrase in SEO history")
print("   - Returns: product_id and product_url")
print()
print("4. give_up:")
print("   - Handles max retries exceeded (MAX_WRITER_ATTEMPTS=5)")
print("   - Returns error message")
print()
print("Routing Logic:")
print("-" * 60)
print("From validator (conditional edges):")
print(f"  - If validation_passed: → publisher")
print(f"  - If writer_attempts ≥ {MAX_WRITER_ATTEMPTS}: → give_up")
print(f"  - Otherwise: → writer (retry)")
print()
print("Exit Points: publisher and give_up both lead to END")

## 7. Run the Workflow with Sample Data

In [ ]:
# Sample product data
sample_input = {
    "input_data": {
        "product_name": "Timer Faucet BPT-09 Self-Closing Basin Tap",
        "raw_specs": "Material: Solid Brass Construction with Polished Chrome Finish; Mechanism: Hydraulic Auto-Shutoff Cartridge; Timing: 7-12 seconds auto-close delay; Operating Pressure: 0.5 Bar to 6.0 Bar; Flow Rate: 3.0 L/min at 3 Bar",
        "category_id": 121,
        "price": "145.00",
        "focus_keyphrase": "Self-Closing Timer Faucet",
        "brand": "MAAT",
        "image_url": "https://maat.ae/wp-content/uploads/2025/09/BPT-08-1.webp",
        "sku": "MAAT-BPT09-TAP"
    },
    "generated_description": {},
    "validation_passed": False,
    "validation_error": "",
    "wordpress_product_id": None,
    "wordpress_product_url": None,
    "internal_link_url": None,
    "writer_attempts": 0
}

print("Sample Input:")
print("=" * 60)
for key, value in sample_input["input_data"].items():
    if key != "raw_specs":
        print(f"{key}: {value}")
    else:
        print(f"{key}: [truncated specs...]")

## 8. Execute the Workflow

In [ ]:
# Run the workflow
print("Executing workflow...")
print("=" * 60)
print()

# Execute the compiled workflow
final_state = fihris_agent.invoke(sample_input)

print("Workflow execution complete!")
print()
print("Final State:")
print("-" * 60)
for key, value in final_state.items():
    if key in ['generated_description', 'input_data']:
        if isinstance(value, dict):
            print(f"{key}:")
            for k, v in value.items():
                print(f"  {k}: {v if len(str(v)) < 100 else str(v)[:97] + '...'}")
        else:
            print(f"{key}: [complex object]")
    else:
        print(f"{key}: {value}")

## 9. Workflow Validation Checks (Production)

The production validator performs these checks:


In [ ]:
validation_checks = {
    "Structural Rules": [
        "No LaTeX formatting ($, \\text)",
        "No forbidden phrases (e.g., 'new arrival')",
        "No Markdown syntax (## headers, ** bold)",
        "Pure HTML only (<p>, <h2>, <h3>, <ul>, <strong>)"
    ],
    "Content Length": [
        "Description must be ≥ 280 words (target ~350)"
    ],
    "Keyphrase - Introduction": [
        "Focus keyphrase must appear in opening paragraph"
    ],
    "Keyphrase - Distribution": [
        "No gap > 35% of sentences without keyphrase/synonym"
    ],
    "Keyphrase - Density": [
        f"Density must be between {MIN_KEYPHRASE_DENSITY}% and {MAX_KEYPHRASE_DENSITY}%",
        "Avoid keyword stuffing"
    ],
    "Keyphrase - Subheading": [
        "At least one <h2>/<h3> must contain the focus keyphrase"
    ],
    "SEO Title": [
        "Must begin with focus keyphrase",
        "Must be ≤ 60 characters"
    ],
    "Meta Description": [
        "Must contain focus keyphrase",
        f"Must be {MIN_META_DESC_LEN}-{MAX_META_DESC_LEN} characters"
    ],
    "Slug": [
        "Must contain slugified focus keyphrase"
    ],
    "Image Alt Text": [
        "Must contain focus keyphrase (if image present)"
    ],
    "Product Attributes": [
        "Must extract ≥ 2 structured attributes from raw specs"
    ],
    "Internal Links": [
        "Must have ≥ 1 internal link to site category"
    ],
    "External Links": [
        "Must have ≥ 1 external link to reputable source",
        "External link must be reachable (URL validation)"
    ]
}

print("PRODUCTION VALIDATION CHECKS")
print("=" * 60)
print()
total_checks = 0
for category, checks in validation_checks.items():
    print(f"📋 {category}:")
    for check in checks:
        print(f"   ✓ {check}")
        total_checks += 1
    print()
print(f"Total validation checks: {total_checks}")

## 10. Workflow State Transitions

Trace the state changes as the workflow progresses:

In [ ]:
# Simulate state transitions with logging
print("STATE TRANSITIONS (Example Flow)")
print("=" * 60)
print()

transitions = [
    {
        "node": "START",
        "action": "Initialize state with product input data",
        "state_updates": {
            "input_data": "Product specifications",
            "writer_attempts": 0,
            "validation_passed": False
        }
    },
    {
        "node": "writer",
        "action": "Generate product copy using Gemini",
        "state_updates": {
            "generated_description": "AI-generated HTML content",
            "writer_attempts": 1,
            "internal_link_url": "Category URL"
        }
    },
    {
        "node": "validator",
        "action": "Validate SEO and content rules",
        "state_updates": {
            "validation_passed": "True/False",
            "validation_error": "Error message if failed"
        }
    },
    {
        "node": "publisher",
        "action": "Publish to WooCommerce (if passed validation)",
        "state_updates": {
            "wordpress_product_id": "Product ID",
            "wordpress_product_url": "Product URL"
        }
    },
    {
        "node": "END",
        "action": "Workflow complete",
        "state_updates": {}
    }
]

for i, transition in enumerate(transitions, 1):
    print(f"Step {i}: {transition['node']}")
    print(f"  Action: {transition['action']}")
    print(f"  State updates:")
    for key, value in transition['state_updates'].items():
        print(f"    • {key}: {value}")
    print()

## 11. Error Handling and Retry Logic

In [ ]:
print("ERROR HANDLING & RETRY LOGIC")
print("=" * 60)
print()
print(f"Maximum writer attempts: {MAX_WRITER_ATTEMPTS}")
print()
print("Retry Scenario:")
print("-" * 60)
print()

retry_flow = [
    "1. writer generates description (attempt 1)",
    "2. validator finds issues (validation_passed=False)",
    "3. routing_logic checks: writer_attempts (1) < MAX_WRITER_ATTEMPTS (5)",
    "4. Flow returns to writer for retry",
    "5. writer generates improved description (attempt 2)",
    "6. validator checks again",
    "...",
    "N. If validation passes before attempt 5 → publisher",
    "N. If attempt 5 fails → give_up node with error message"
]

for step in retry_flow:
    print(f"  {step}")
print()
print("This allows up to 5 attempts for the AI to generate valid content.")

## 12. Key Configuration Parameters

In [ ]:
print("CONFIGURATION PARAMETERS")
print("=" * 60)
print()

config_params = {
    "MAX_WRITER_ATTEMPTS": f"{MAX_WRITER_ATTEMPTS} - Maximum retry attempts",
    "MIN_KEYPHRASE_DENSITY": f"{MIN_KEYPHRASE_DENSITY}% - Minimum keyword density",
    "MAX_KEYPHRASE_DENSITY": f"{MAX_KEYPHRASE_DENSITY}% - Maximum keyword density (avoid stuffing)",
    "MIN_META_DESC_LEN": f"{MIN_META_DESC_LEN} chars - Minimum meta description length",
    "MAX_META_DESC_LEN": f"{MAX_META_DESC_LEN} chars - Maximum meta description length"
}

for param, description in config_params.items():
    print(f"{param}: {description}")
print()
print("All parameters are defined in app/graph.py and can be adjusted as needed.")

## Summary

This notebook demonstrates the complete LangGraph workflow for the Fihris Agent:

- **Workflow nodes**: writer, validator, publisher, give_up
- **Conditional routing**: Based on validation results and retry count
- **Validation**: 15+ SEO and content quality checks
- **Retry logic**: Up to 5 attempts to generate valid content
- **Integration**: Publishes to WooCommerce API on success

The workflow handles complex product data transformation and ensures SEO compliance before publication.